# Pipeline metagenómico por muestra — versión Notebook

De FASTQ crudo (ENA) hasta genes de resistencia (RGI/CARD), paso a paso y de forma interactiva.

**Cómo usar este cuaderno**
- Colócalo en la **raíz del repositorio** (el mismo sitio donde está `run_pipeline.sh`) y ejecútalo desde ahí, porque usa rutas relativas (`raw/`, `work/`, `results/`).
- Necesitas el entorno `amr-ml` activo (lánzalo con `jupyter lab` desde ese entorno) y la base **CARD ya cargada** en esta carpeta: `rgi load --card_json card.json --local`.
- Cada paso es *idempotente*: si su salida ya existe, se salta. Puedes re-ejecutar tras un corte sin repetir trabajo.

> Para procesar **muchas** muestras en lote, el script `run_pipeline.sh` sigue siendo mejor (se lanza en bucle y no depende de tener el navegador abierto). Este cuaderno es para correr **una** muestra de forma interactiva e inspeccionar los resultados.

## 0. Configuración y utilidades

In [ ]:
import os, subprocess, hashlib, shutil
from pathlib import Path
import pandas as pd

# ---- Parámetros (edita aquí) ----
RUN       = "ERR1135202"    # run_accession de ENA a procesar
SUBSAMPLE = 1_000_000        # nº de pares a submuestrear (ensayo rápido)
THREADS   = 4                # hilos de CPU
SEED      = 100              # semilla fija -> submuestreo reproducible

# ---- Carpetas de trabajo (se crean si no existen) ----
RAW_DIR  = Path(f"raw/{RUN}")      # FASTQ descargados
WORK_DIR = Path(f"work/{RUN}")     # intermedios
OUT_DIR  = Path(f"results/{RUN}")  # salida final
for d in (RAW_DIR, WORK_DIR, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

def sh(cmd):
    """Ejecuta un comando de shell, muestra su salida y aborta si falla."""
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if r.stdout:
        print(r.stdout)
    if r.returncode != 0:
        raise RuntimeError(f"Falló (código {r.returncode}): {cmd}")
    return r.stdout

print("Entorno listo. Muestra a procesar:", RUN)

## 1. Resolver URLs y checksums desde la API de ENA

En vez de adivinar rutas FTP, le preguntamos a ENA por los archivos de este run y sus MD5. `pandas` puede leer la respuesta directamente desde la URL.

In [ ]:
ena_url = (
    "https://www.ebi.ac.uk/ena/portal/api/filereport"
    f"?accession={RUN}&result=read_run"
    "&fields=fastq_ftp,fastq_md5&format=tsv"
)
ena = pd.read_csv(ena_url, sep="\t")
ena

In [ ]:
# Cada campo trae "archivo_1;archivo_2" (paired-end). Los separamos.
ftp = ena.loc[0, "fastq_ftp"].split(";")
md5 = ena.loc[0, "fastq_md5"].split(";")
url1, url2 = "https://" + ftp[0], "https://" + ftp[1]
r1 = RAW_DIR / f"{RUN}_1.fastq.gz"
r2 = RAW_DIR / f"{RUN}_2.fastq.gz"
print(url1)
print(url2)

## 2. Descargar (reanudable) y verificar integridad con MD5

`wget -c` reanuda si la descarga se corta. Después comprobamos el MD5 en Python; si no coincide, abortamos (una descarga corrupta causa errores fantasma más adelante).

In [ ]:
def md5sum(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

def download(url, out, want_md5):
    out = Path(out)
    if out.exists() and md5sum(out) == want_md5:
        print(f"Ya existe y el MD5 coincide: {out.name}")
        return
    sh(f"wget -c -q --tries=0 --read-timeout=30 --waitretry=5 -O {out} {url}")
    got = md5sum(out)
    if got != want_md5:
        raise RuntimeError(f"MD5 no coincide para {out.name}: {got} != {want_md5}")
    print(f"Descarga verificada: {out.name}")

download(url1, r1, md5[0])
download(url2, r2, md5[1])

## 3. Control de calidad y recorte de adaptadores (fastp)

Genera además un reporte visual `.html` que puedes abrir para ver el antes/después.

In [ ]:
clean1 = WORK_DIR / "clean_1.fastq.gz"
clean2 = WORK_DIR / "clean_2.fastq.gz"
if clean1.exists() and clean2.exists():
    print("QC ya realizado, se omite")
else:
    sh(f"fastp -i {r1} -I {r2} -o {clean1} -O {clean2} "
       f"-w {THREADS} -h {OUT_DIR}/{RUN}_fastp.html -j {OUT_DIR}/{RUN}_fastp.json")

## 4. Submuestreo reproducible (seqtk)

Tomamos una fracción al azar con la **misma semilla** en ambos pares, para que las lecturas sigan emparejadas. Esto hace el ensayo rápido; para la muestra completa, sube `SUBSAMPLE` o sáltate este paso.

In [ ]:
sub1 = WORK_DIR / "sub_1.fastq"
sub2 = WORK_DIR / "sub_2.fastq"
if sub1.exists() and sub2.exists():
    print("Submuestreo ya realizado, se omite")
else:
    sh(f"seqtk sample -s{SEED} {clean1} {SUBSAMPLE} > {sub1}")
    sh(f"seqtk sample -s{SEED} {clean2} {SUBSAMPLE} > {sub2}")

## 5. Ensamblado (MEGAHIT)

Reconstruye contigs a partir de las lecturas. **Este paso es el más lento** (varios minutos); en el notebook su salida aparecerá cuando termine, no en vivo. Ten paciencia.

In [ ]:
contigs = WORK_DIR / "megahit_out" / "final.contigs.fa"
if contigs.exists():
    print("Ensamblado ya existe, se omite")
else:
    shutil.rmtree(WORK_DIR / "megahit_out", ignore_errors=True)  # MEGAHIT exige que NO exista
    sh(f"megahit -1 {sub1} -2 {sub2} -t {THREADS} -o {WORK_DIR}/megahit_out")

## 6. Predicción de genes (Prodigal)

Encuentra los genes dentro de los contigs y los guarda como proteínas (`.faa`) y nucleótidos (`.fna`).

In [ ]:
genes_faa = WORK_DIR / "genes.faa"
genes_fna = WORK_DIR / "genes.fna"
if not genes_faa.exists():
    sh(f"prodigal -i {contigs} -a {genes_faa} -d {genes_fna} -p meta -q")

n_genes = int(subprocess.run(f"grep -c '>' {genes_faa}", shell=True,
                             text=True, capture_output=True).stdout or 0)
print(f"Genes predichos: {n_genes}")

## 7. Detección de genes de resistencia (RGI contra CARD)

Usamos las proteínas (`-t protein`) porque ya las predijo Prodigal. Requiere la base CARD cargada localmente.

In [ ]:
if not (Path("localDB/card.json").exists() or Path("card.json").exists()):
    raise RuntimeError(
        "No encuentro la base CARD local. Corre antes en esta carpeta:\n"
        "  rgi load --card_json card.json --local"
    )

rgi_out = OUT_DIR / f"rgi_{RUN}"
sh(f"rgi main -i {genes_faa} -o {rgi_out} -t protein -a DIAMOND --local --clean")

## 8. Cargar y visualizar el resistoma

Aquí es donde el notebook brilla: leemos la tabla de RGI con pandas y la resumimos. Estas columnas son las que alimentarán después el modelo (familia de ARG, clase de antibiótico, mecanismo, y el identificador ARO).

In [ ]:
from Bio import SeqIO

rgi = pd.read_csv(f"{rgi_out}.txt", sep="\t")
print(f"Genes de resistencia detectados: {len(rgi)}")
print("Por tipo de acierto (Cut_Off):")
print(rgi["Cut_Off"].value_counts())

# Posición de cada ARG dentro de su contig: RGI ya trae Start/Stop/Orientation (coordenadas de
# nucleótido dentro del contig, heredadas del encabezado que deja Prodigal), pero no el largo del
# contig -- lo sacamos del ensamblado para poder expresar una posición relativa (0 = inicio del
# contig, 1 = final), útil para comparar genes entre contigs de distinto tamaño.
largos_contig = {r.id: len(r.seq) for r in SeqIO.parse(str(contigs), "fasta")}
rgi["contig_len"] = rgi["Contig"].map(largos_contig)
rgi["pos_inicio_rel"] = rgi["Start"] / rgi["contig_len"]
rgi["pos_fin_rel"] = rgi["Stop"] / rgi["contig_len"]

rgi[["Contig", "Start", "Stop", "Orientation", "contig_len", "pos_inicio_rel", "pos_fin_rel",
     "Best_Hit_ARO", "Drug Class", "Resistance Mechanism", "AMR Gene Family"]]

In [ ]:
import matplotlib.pyplot as plt

fam = rgi["AMR Gene Family"].value_counts()
ax = fam.plot(kind="barh")
ax.invert_yaxis()
plt.xlabel("Nº de genes detectados")
plt.title(f"Familias de genes de resistencia — {RUN}")
plt.tight_layout()
plt.show()

---
**Siguiente paso.** Este resistoma es el lado de la *etiqueta* para el modelo (qué familias de ARG hay, y en qué
posición de su contig). Lo que falta es el *quién*: a qué bacteria pertenece cada gen (asignación taxonómica) y en
qué posición de su genoma de referencia (BLAST contra `nt` también da esas coordenadas, no solo la especie), para
luego cruzarlo con los rasgos de BacDive. Ese flujo completo -- con BLAST y la proyección de posición sobre la
referencia -- está en `04_pipeline_organizado.ipynb`, sección 3.6.